# 🔧 Rossmann Retail Sales Forecasting
## Notebook 2: Feature Engineering

**Author:** Cephas Adams Kumah  
**Dataset:** Rossmann Store Sales (Kaggle)  
**Input:** `train.csv` + `store.csv` (844,338 trading-day records)  
**Output:** `sales_features.csv` (810,888 records · 36 columns)

---

### Notebook Workflow

| Step | Description |
|---|---|
| **1. Load & Merge Data** | Load the raw datasets, handle missing values, and merge store information with daily sales records |
| **2. Create Date Features** | Extract calendar-based features to capture seasonality and trading patterns |
| **3. Generate Lag Features** | Create historical sales features using 7-, 14-, and 28-day lags |
| **4. Compute Rolling Statistics** | Calculate rolling averages and variability to capture recent sales momentum |
| **5. Encode Categorical Variables** | Encode store attributes and holiday categories for machine learning models |
| **6. Export Feature Dataset** | Remove incomplete records created during feature generation and save the final modelling dataset |

## 1. Load and Merge Data

Both raw datasets are loaded and prepared before merging. The preparation process includes three key steps:

1. **Filter to trading days only:** Records where `Open = 0` or `Sales = 0` are removed because they represent closed stores or non-trading days.
2. **Handle missing store attributes:** Missing `CompetitionDistance` values are filled with the median distance, while missing competition and promotion timing fields are filled with `0` to indicate unavailable history.
3. **Merge on Store ID:** The sales data is left-joined with store-level attributes using the `Store` column.

This produces a merged dataset of **844,338 open trading-day records** ready for feature engineering.

In [1]:
import pandas as pd
import numpy as np

# Load raw datasets
train = pd.read_csv('../data/train.csv', low_memory=False)
store = pd.read_csv('../data/store.csv')

# Prepare sales data for analysis
train['Date'] = pd.to_datetime(train['Date'])
train = train[(train['Open'] == 1) & (train['Sales'] > 0)]

# Handle missing store-level values
store['CompetitionDistance'] = store['CompetitionDistance'].fillna(
    store['CompetitionDistance'].median()
)

fill_zero_cols = [
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2SinceWeek',
    'Promo2SinceYear'
]

store[fill_zero_cols] = store[fill_zero_cols].fillna(0)
store['PromoInterval'] = store['PromoInterval'].fillna('None')

# Merge sales records with store attributes
df = train.merge(store, on='Store', how='left')

print(f"Loaded: {df.shape}")

Loaded: (844338, 18)


## 2. Extract Date Features

Calendar features are created to capture time-based sales patterns that tree-based models cannot learn from raw date values alone.

| Feature | Type | Captures |
|---|---|---|
| `Year` | Integer | Long-term sales trends |
| `Month` | Integer, 1 to 12 | Monthly seasonality and demand cycles |
| `Day` | Integer, 1 to 31 | Pay-day, mid-month, and month-end effects |
| `WeekOfYear` | Integer, 1 to 52 | Weekly seasonality and holiday-period patterns |
| `Quarter` | Integer, 1 to 4 | Quarterly business patterns |
| `IsWeekend` | Binary | Weekend trading behaviour |
| `IsMonthStart` | Binary | Start-of-month shopping behaviour |
| `IsMonthEnd` | Binary | End-of-month shopping behaviour |

In [5]:
# Create calendar-based features for trend and seasonality analysis
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['Date'].dt.quarter

#Add useful business timing indicators
df['IsWeekend'] = df['DayOfWeek'].isin([6,7]).astype(int)
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)

print("Date features created")

print(df[['Date', 'Year', 'Month', 'Day', 'WeekOfYear',
          'Quarter', 'IsWeekend']].head(5))

Date features created
        Date  Year  Month  Day  WeekOfYear  Quarter  IsWeekend
0 2015-07-31  2015      7   31          31        3          0
1 2015-07-31  2015      7   31          31        3          0
2 2015-07-31  2015      7   31          31        3          0
3 2015-07-31  2015      7   31          31        3          0
4 2015-07-31  2015      7   31          31        3          0


## 3.Create Lag Features

Lag features capture historical sales for each store and provide the strongest signal for short-term demand forecasting. They enable the model to learn recurring sales patterns from previous trading periods.

| Feature | Looks Back | Business Meaning |
|---|---:|---|
| `Sales_Lag_7` | 7 trading days | Sales on the same day one week earlier |
| `Sales_Lag_14` | 14 trading days | Sales on the same day two weeks earlier |
| `Sales_Lag_28` | 28 trading days | Sales on the same day four weeks earlier |

> ⚠️ **Leakage Prevention:** Lag features are generated separately for each store using `groupby('Store').shift(n)`. This ensures that each prediction is based only on historical sales from the same store and prevents future observations from leaking into the model.

In [6]:
# Sort records before creating time-based features
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

## Create store-level lag features from previous trading periods
df['Sales_Lag_7'] = df.groupby('Store')['Sales'].shift(7)
df['Sales_Lag_14'] = df.groupby('Store')['Sales'].shift(14)
df['Sales_Lag_28'] = df.groupby('Store')['Sales'].shift(28)

print("Lag features created:")
print(df[['Store', 'Date', 'Sales', 'Sales_Lag_7', 
          'Sales_Lag_14', 'Sales_Lag_28']].head(35))

Lag features created:
    Store       Date  Sales  Sales_Lag_7  Sales_Lag_14  Sales_Lag_28
0       1 2013-01-02   5530          NaN           NaN           NaN
1       1 2013-01-03   4327          NaN           NaN           NaN
2       1 2013-01-04   4486          NaN           NaN           NaN
3       1 2013-01-05   4997          NaN           NaN           NaN
4       1 2013-01-07   7176          NaN           NaN           NaN
5       1 2013-01-08   5580          NaN           NaN           NaN
6       1 2013-01-09   5471          NaN           NaN           NaN
7       1 2013-01-10   4892       5530.0           NaN           NaN
8       1 2013-01-11   4881       4327.0           NaN           NaN
9       1 2013-01-12   4952       4486.0           NaN           NaN
10      1 2013-01-14   4717       4997.0           NaN           NaN
11      1 2013-01-15   3900       7176.0           NaN           NaN
12      1 2013-01-16   4008       5580.0           NaN           NaN
13      1 20

## 4. Create Rolling Average Features

Rolling statistics smooth out daily sales fluctuations by summarizing recent sales behaviour over fixed time windows. These features help the model capture sales momentum, showing whether a store’s sales are trending upward, trending downward, or becoming more volatile.

| Feature | Window | Captures |
|---|---:|---|
| `Rolling_Mean_7` | 7 trading records | Short-term sales momentum |
| `Rolling_Mean_30` | 30 trading records | Medium-term sales trend |
| `Rolling_Std_7` | 7 trading records | Recent sales volatility |

> ⚠️ **Leakage Prevention:** `.shift(1)` is applied before `.rolling()` so the current day’s sales are excluded from the rolling calculation. This ensures that each feature is based only on historical sales data available before the prediction date.

In [13]:
# Create store-level rolling features using only past sales values
df['Rolling_Mean_7'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)
df['Rolling_Mean_30'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(30).mean()
)
df['Rolling_Std_7'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(7).std()
)

print("Rolling feateures created:")
print(df[['Store','Date','Sales',
        'Rolling_Mean_7','Rolling_Mean_30']].head(35))

Rolling feateures created:
    Store       Date  Sales  Rolling_Mean_7  Rolling_Mean_30
0       1 2013-01-02   5530             NaN              NaN
1       1 2013-01-03   4327             NaN              NaN
2       1 2013-01-04   4486             NaN              NaN
3       1 2013-01-05   4997             NaN              NaN
4       1 2013-01-07   7176             NaN              NaN
5       1 2013-01-08   5580             NaN              NaN
6       1 2013-01-09   5471             NaN              NaN
7       1 2013-01-10   4892     5366.714286              NaN
8       1 2013-01-11   4881     5275.571429              NaN
9       1 2013-01-12   4952     5354.714286              NaN
10      1 2013-01-14   4717     5421.285714              NaN
11      1 2013-01-15   3900     5381.285714              NaN
12      1 2013-01-16   4008     4913.285714              NaN
13      1 2013-01-17   4044     4688.714286              NaN
14      1 2013-01-18   4127     4484.857143              N

## 5. Encode Categorical Variables

Three categorical variables are label encoded to make them suitable for machine learning models. Label encoding replaces each category with a unique integer value. This approach is appropriate for tree-based algorithms such as Random Forest and XGBoost because they split data based on feature values rather than assuming a numerical or ordinal relationship between categories.

| Original Column | Encoded Column | Encoding |
|---|---|---|
| `StoreType` | `StoreType_enc` | a → 0, b → 1, c → 2, d → 3 |
| `Assortment` | `Assortment_enc` | a → 0, b → 1, c → 2 |
| `StateHoliday` | `StateHoliday_enc` | 0 → 0, a → 1, b → 2, c → 3 |

The encoded variables preserve the original categorical information while enabling efficient model training and prediction.

In [16]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical features for machine learning
le = LabelEncoder()
df['StoreType_enc']    = le.fit_transform(df['StoreType'])
df['Assortment_enc']   = le.fit_transform(df['Assortment'])
df['StateHoliday_enc'] = le.fit_transform(df['StateHoliday'].astype(str))

# Review encoded category mappings
print("Categorical columns encoded.")
print(df[['StoreType','StoreType_enc',
          'Assortment','Assortment_enc']].drop_duplicates())

Categorical columns encoded.
       StoreType  StoreType_enc Assortment  Assortment_enc
0              c              2          a               0
781            a              0          a               0
2344           c              2          c               2
4687           a              0          c               2
9388           d              3          a               0
10788          d              3          c               2
63445          b              1          a               0
194128         b              1          b               1
425006         b              1          c               2


## 6. Drop Missing Lag and Rolling Feature Rows

The first several records for each store contain missing values in the lag and rolling feature columns because there is not enough historical data available to calculate those features. These incomplete rows are removed before saving the final modelling dataset.

| Stage | Records |
|---|---:|
| After filtering closed days | 844,338 |
| After removing missing lag and rolling feature rows | **810,888** |
| Records removed | 33,450 (4.0%) |

The final feature-engineered dataset is saved as **`sales_features.csv`** and used directly in **Notebook 3** for model training and evaluation.

In [19]:
# Remove rows without enough historical sales data for lag/rolling features
required_features = [
    'Sales_Lag_7',
    'Sales_Lag_14',
    'Sales_Lag_28',
    'Rolling_Mean_7',
    'Rolling_Mean_30'
]

df_clean = df.dropna(subset=required_features)

print(f"Rows before dropping NaN: {len(df):,}")
print(f"Rows after dropping NaN: {len(df_clean):,}")

# Save final feature-engineered dataset for modeling
df_clean.to_csv('../data/sales_features.csv', index=False)
print(f"\nFeature dataset saved: {df_clean.shape}")

Rows before dropping NaN: 844,338
Rows after dropping NaN: 810,888

Feature dataset saved: (810888, 36)


---
## ✅ Feature Engineering Complete

The feature engineering pipeline successfully transformed the raw sales and store datasets into a modelling-ready dataset.

**Output file:** `sales_features.csv`  
**Dataset size:** **810,888 records × 36 columns**

### Dataset Summary

| Stage | Records |
|---|---:|
| Trading-day records before feature generation | 844,338 |
| Final modelling dataset | **810,888** |
| Records removed | 33,450 (4.0%) |

### Engineered Features

| Feature Group | Count |
|---|---:|
| Lag Features | 3 |
| Rolling Statistics | 3 |
| Calendar Features | 8 |
| Encoded Categorical Features | 3 |
| **Total Engineered Features** | **17** |

The final dataset is exported as **`sales_features.csv`** and serves as the input to **Notebook 3**, where Random Forest and XGBoost models are trained, evaluated, and compared.